# 02 · Data Understanding (1/2) — Pengumpulan Data (Scraping)
Fase CRISP-DM 2, tugas **Collect Initial Data** → *initial data collection report*.

| Sumber | Isi | Akses | Dipakai untuk |
|---|---|---|---|
| **PIHPS Nasional – Bank Indonesia** (`bi.go.id/hargapangan`) | Harga harian 21 varian (10 komoditas), 4 jenis pasar, 34 provinsi + nasional, 2018–kini | Publik, gratis, endpoint JSON | Target & fitur utama |
| **NASA POWER** (`power.larc.nasa.gov`) | Curah hujan harian terkoreksi (PRECTOTCORR) di ibu kota 34 provinsi, 1991–kini | Publik, gratis, API tanpa key | Fitur anomali hujan |
| **Kalender hari besar** | 1 Ramadan, Idulfitri, Iduladha (hasil sidang isbat/SKB), Natal, Tahun Baru | Ditulis di `reference.py` | Fitur kalender |
| **geoBoundaries IDN ADM1** | Batas 34 provinsi (© OpenStreetMap, ODbL) | Unduh sekali, disederhanakan | Peta dashboard |

In [1]:
# --- Setup: path project, modul radarpangan, gaya grafik -----------------------------------
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from radarpangan.config import load_project
cfg, P = load_project(ROOT)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .3, "font.size": 10})
print("Project:", ROOT)

Project: /home/claude/MiningProcess


## 2.1 Membongkar API PIHPS

Halaman **Tabel Harga** PIHPS (`/hargapangan/TabelHarga/PasarTradisionalKomoditas`, dst.) dibangun dengan komponen DevExtreme yang memanggil endpoint JSON berikut (ditemukan dengan menginspeksi kode halaman / *Network tab* browser):

| Endpoint (`/hargapangan/WebSite/TabelHarga/…`) | Fungsi |
|---|---|
| `GetRefPriceType` | Jenis pasar: 1 Pasar Tradisional, 2 Pasar Modern, 3 Pedagang Besar, 4 Produsen |
| `GetRefCommodityAndCategory` | Pohon komoditas: kategori `cat_1..cat_10`, varian `com_1..com_21` |
| `GetRefProvince`, `GetRefRegency` | Daftar provinsi (id 1–34) & kab/kota |
| **`GetGridDataKomoditas`** | **Tabel harga 1 komoditas: baris = nasional + semua provinsi, kolom = tanggal** |

Parameter `GetGridDataKomoditas`: `price_type_id`, `comcat_id` (mis. `com_14`), `province_id` (kosong = semua), `regency_id`, `showKota`, `showPasar`, `tipe_laporan` (1 = harian), `start_date`, `end_date` (`YYYY-MM-DD`).

Temuan penting saat uji coba:
- Rentang panjang membuat server sangat lambat (1 bulan ≈ 2 detik, 1 tahun ≈ 45 detik) → **scraping per bulan**.
- Kunci tanggal berformat `DD/MM/YYYY`, **hanya hari kerja**; harga berupa teks `"67,000"`; nilai kosong ditulis `"-"`.
- Data 34 provinsi lengkap mulai **Januari 2018** (2017 hanya sebagian provinsi).

In [2]:
from radarpangan.scraper import PIHPSClient, save_reference_tables, parse_grid, scrape_pihps, load_raw_cache, month_chunks
from radarpangan.pipeline import client_from_cfg

client = client_from_cfg(cfg)
refs = save_reference_tables(client, P.external)          # simpan ke data/external/*.csv
display(refs["pihps_price_types"])
com = refs["pihps_commodities"]
display(com[com["id"].str.startswith("com_")].merge(
    com[com["id"].str.startswith("cat_")][["id", "name"]].rename(columns={"id": "cat_id", "name": "kategori"}), on="cat_id"
)[["id", "name", "kategori", "denomination"]].rename(columns={"id": "varian_id", "name": "varian"}))
print("Jumlah provinsi:", len(refs["pihps_provinces"]))

,id,name
0,1,Pasar Tradisional
1,2,Pasar Modern
2,3,Pedagang Besar
3,4,Produsen


,varian_id,varian,kategori,denomination
0,com_1,Beras Kualitas Bawah I,Beras,kg
1,com_2,Beras Kualitas Bawah II,Beras,kg
2,com_3,Beras Kualitas Medium I,Beras,kg
3,com_4,Beras Kualitas Medium II,Beras,kg
4,com_5,Beras Kualitas Super I,Beras,kg
5,com_6,Beras Kualitas Super II,Beras,kg
6,com_7,Daging Ayam Ras Segar,Daging Ayam,kg
7,com_8,Daging Sapi Kualitas 1,Daging Sapi,kg
8,com_9,Daging Sapi Kualitas 2,Daging Sapi,kg
9,com_10,Telur Ayam Ras Segar,Telur Ayam,kg


Jumlah provinsi: 34


### Contoh 1 request: Cabai Merah Keriting (`com_14`), Pasar Tradisional, 1 bulan terakhir

In [3]:
end = pd.Timestamp.today().normalize()
start = (end - pd.Timedelta(days=30))
records = client.grid_komoditas(price_type_id=1, comcat_id="com_14", start=start, end=end)
print("Jumlah baris (nasional + provinsi):", len(records))
print(json.dumps(records[:2], ensure_ascii=False)[:700], "...")
sample = parse_grid(records, 1, "com_14")
display(sample.head(8))

Jumlah baris (nasional + provinsi): 35
[{"no": "I", "name": "Semua Provinsi", "level": 0, "28/08/2026": "53,250", "31/08/2026": "55,850", "01/09/2026": "56,550", "02/09/2026": "56,950", "03/09/2026": "57,600", "04/09/2026": "57,750", "07/09/2026": "60,250", "08/09/2026": "61,000", "09/09/2026": "61,600", "10/09/2026": "62,300", "11/09/2026": "63,150", "14/09/2026": "66,450", "15/09/2026": "67,000", "16/09/2026": "68,650", "17/09/2026": "68,700", "18/09/2026": "69,150", "21/09/2026": "70,050", "22/09/2026": "69,800", "23/09/2026": "68,750", "24/09/2026": "69,350", "25/09/2026": "69,250"}, {"no": "II", "name": "Aceh", "level": 1, "28/08/2026": "45,850", "31/08/2026": "52,600", "01/09/2026": "56,850", "02/09/2026": "60,400", "03/09/ ...


,date,price_type_id,commodity_id,region_name,level,price
0,2026-08-28,1,com_14,Semua Provinsi,0,"53,250.0000"
1,2026-08-31,1,com_14,Semua Provinsi,0,"55,850.0000"
2,2026-09-01,1,com_14,Semua Provinsi,0,"56,550.0000"
3,2026-09-02,1,com_14,Semua Provinsi,0,"56,950.0000"
4,2026-09-03,1,com_14,Semua Provinsi,0,"57,600.0000"
5,2026-09-04,1,com_14,Semua Provinsi,0,"57,750.0000"
6,2026-09-07,1,com_14,Semua Provinsi,0,"60,250.0000"
7,2026-09-08,1,com_14,Semua Provinsi,0,"61,000.0000"


## 2.2 Scraping penuh (4 jenis pasar × 21 varian × Jan 2018 – hari ini)

- Total ± **8.800 request** (4 × 21 × ±105 bulan). Dengan 2–4 *worker*, run pertama ± 40–90 menit.
- Setiap chunk disimpan apa adanya (`data/raw/pihps/pt{jenis}/{varian}/{YYYY-MM}.json.gz`) → proses bisa dihentikan & dilanjutkan kapan saja.
- **2 bulan terakhir selalu diambil ulang**, karena data terbaru PIHPS bisa direvisi/bertambah.
- Etika: jeda antar-request, jumlah worker kecil, User-Agent jujur, data publik untuk riset non-komersial.

In [4]:
RUN_SCRAPE = True          # set False jika hanya ingin memakai cache yang sudah ada
sc = cfg["scraping"]
all_coms = [c for c in com["id"] if c.startswith("com_")]
n_chunks = len(sc["price_types"]) * len(all_coms) * len(month_chunks(sc["start_date"], sc["end_date"] or pd.Timestamp.today()))
n_cached = len(list(P.raw_pihps.glob("pt*/*/*.json.gz")))
print(f"Total chunk: {n_chunks:,} | sudah ada di cache: {n_cached:,}")

if RUN_SCRAPE:
    summary = scrape_pihps(client, P.raw_pihps, sc["price_types"], all_coms, sc["start_date"], sc["end_date"],
                           workers=sc["workers"], refresh_recent_months=sc["refresh_recent_months"])
    display(summary["status"].value_counts().rename("jumlah chunk").to_frame())
    if (summary["status"] == "error").any():
        display(summary[summary["status"] == "error"].head())

Total chunk: 8,820 | sudah ada di cache: 8,820


Scraping PIHPS: 100%|██████████| 168/168 [02:29<00:00,  1.13chunk/s]


,jumlah chunk
status,
cached,8652
ok,168


## 2.3 Menggabungkan cache menjadi satu tabel "long"
Satu baris = (tanggal, jenis pasar, varian, provinsi, harga). Baris `"Semua Provinsi"` disimpan sebagai provinsi `0 = Nasional`.

In [5]:
long = load_raw_cache(P.raw_pihps, refs["pihps_provinces"], refs["pihps_commodities"])
long.to_parquet(P.interim / "pihps_long.parquet", index=False)
print(f"{len(long):,} baris | {long['date'].min():%d %b %Y} – {long['date'].max():%d %b %Y}")
display(long.sample(5, random_state=1))
long.info(memory_usage="deep")

Nama wilayah tidak dikenal (dibuang): ['Dum']


5,450,980 baris | 01 Jan 2018 – 25 Sep 2026


,date,price_type_id,price_type,commodity_id,commodity,cat_id,category,province_id,province,price
3886388,2021-07-15,3,Pedagang Besar,com_20,Gula Pasir Kualitas Premium,cat_10,Gula Pasir,20,Kalimantan Barat,"12,500.0000"
3411674,2018-04-19,3,Pedagang Besar,com_15,Cabai Rawit Hijau,cat_8,Cabai Rawit,4,Riau,"34,150.0000"
2669072,2020-12-29,2,Pasar Modern,com_5,Beras Kualitas Super I,cat_1,Beras,23,Kalimantan Timur,"13,400.0000"
1939785,2025-10-09,2,Pasar Modern,com_13,Cabai Merah Besar,cat_7,Cabai Merah,24,Kalimantan Utara,"88,000.0000"
1388299,2018-05-22,1,Pasar Tradisional,com_6,Beras Kualitas Super II,cat_1,Beras,33,Papua,"12,900.0000"


<class 'pandas.DataFrame'>
RangeIndex: 5450980 entries, 0 to 5450979
Data columns (total 10 columns):
 #   Column         Dtype         
---  ------         -----         
 0   date           datetime64[us]
 1   price_type_id  int64         
 2   price_type     str           
 3   commodity_id   str           
 4   commodity      str           
 5   cat_id         str           
 6   category       str           
 7   province_id    int64         
 8   province       str           
 9   price          float64       
dtypes: datetime64[us](1), float64(1), int64(2), str(6)
memory usage: 772.7 MB


## 2.4 Curah hujan harian NASA POWER
Titik = koordinat ibu kota/kota pasar utama tiap provinsi. Diambil sejak 1991 supaya **klimatologi (1991–2017)** bisa dihitung dari periode *sebelum* data harga → fitur anomali bebas kebocoran.

In [6]:
from radarpangan.reference import province_reference, holiday_table
from radarpangan.weather import fetch_all_provinces

ref = province_reference()
ref.to_csv(P.external / "province_reference.csv", index=False)
rain = fetch_all_provinces(ref, P.raw_weather, cfg["weather"]["climatology_start"])
rain.to_parquet(P.interim / "rainfall_daily.parquet", index=False)
print(f"{len(rain):,} baris | {rain['date'].min():%Y-%m-%d} – {rain['date'].max():%Y-%m-%d} | "
      f"kosong: {rain['precip_mm'].isna().mean():.2%} (NASA POWER ~3 hari terakhir belum tersedia)")
display(ref.head())

443,836 baris | 1991-01-01 – 2026-09-27 | kosong: 0.02% (NASA POWER ~3 hari terakhir belum tersedia)


,province_id,province,city,lat,lon,island
0,1,Aceh,Banda Aceh,5.5483,95.3238,Sumatera
1,2,Sumatera Utara,Medan,3.5952,98.6722,Sumatera
2,3,Sumatera Barat,Padang,-0.9471,100.4172,Sumatera
3,4,Riau,Pekanbaru,0.5071,101.4478,Sumatera
4,5,Kepulauan Riau,Tanjung Pinang,0.9186,104.4665,Sumatera


## 2.5 Kalender hari besar
Tanggal resmi pemerintah (hasil sidang isbat/SKB) untuk 2017–2026; tahun setelahnya diestimasi dengan kalender Hijriah Umm al-Qura (selisih maks. ±1 hari).

In [7]:
hol = holiday_table(2017, 2028)
hol.to_csv(P.external / "hari_besar.csv", index=False)
h3 = hol[hol["event"].isin(["awal_ramadan", "idulfitri", "iduladha"])].assign(tahun=lambda d: d["date"].dt.year)
display(h3.pivot(index="tahun", columns="event", values="date")[["awal_ramadan", "idulfitri", "iduladha"]]
          .apply(lambda s: s.dt.strftime("%d %b %Y")))

event,awal_ramadan,idulfitri,iduladha
tahun,,,
2017,27 May 2017,25 Jun 2017,01 Sep 2017
2018,17 May 2018,15 Jun 2018,22 Aug 2018
2019,06 May 2019,05 Jun 2019,11 Aug 2019
2020,24 Apr 2020,24 May 2020,31 Jul 2020
2021,13 Apr 2021,13 May 2021,20 Jul 2021
2022,03 Apr 2022,02 May 2022,10 Jul 2022
2023,23 Mar 2023,22 Apr 2023,29 Jun 2023
2024,12 Mar 2024,10 Apr 2024,17 Jun 2024
2025,01 Mar 2025,31 Mar 2025,06 Jun 2025


## 2.6 Laporan pengumpulan data (*initial data collection report*)

In [8]:
rep = (long.groupby("price_type", observed=True)
           .agg(baris=("price", "size"), harga_terisi=("price", "count"), tanggal_awal=("date", "min"),
                tanggal_akhir=("date", "max"), n_provinsi=("province_id", lambda s: s[s > 0].nunique()),
                n_varian=("commodity_id", "nunique")))
rep["persen_terisi"] = rep["harga_terisi"] / rep["baris"]
display(rep)
rep.to_csv(P.tables / "02_laporan_pengumpulan_data.csv")
print("Ukuran cache mentah:", round(sum(f.stat().st_size for f in P.raw_pihps.rglob('*.json.gz')) / 1e6, 1), "MB")

,baris,harga_terisi,tanggal_awal,tanggal_akhir,n_provinsi,n_varian,persen_terisi
price_type,,,,,,,
Pasar Modern,1316521,1286636,2018-01-01,2026-09-25,34,21,0.9773
Pasar Tradisional,1627555,1569873,2018-01-01,2026-09-25,34,21,0.9646
Pedagang Besar,1582650,1566791,2018-01-01,2026-09-25,34,21,0.9900
Produsen,924254,901570,2018-03-01,2026-09-25,33,21,0.9755


Ukuran cache mentah: 19.1 MB


### Ringkasan fase 2 (bagian 1)
- API PIHPS berhasil dipetakan; scraper modular dengan cache per bulan, *retry*, dan *resume*.
- Terkumpul data harga harian **4 jenis pasar × 21 varian × 34 provinsi + nasional, Jan 2018 – kini**, ditambah curah hujan NASA POWER 1991–kini, kalender hari besar, dan peta provinsi.
- Output: `data/raw/…`, `data/interim/pihps_long.parquet`, `data/interim/rainfall_daily.parquet`, `data/external/*.csv`.

➡️ Lanjut ke **`03_data_understanding_eda.ipynb`**.